# Breast cancer drug response with PrimeKG

Compare **HER2-positive** and **triple-negative** breast cancer cell lines. This version uses measured GDSC drug response as the training outcome. It reuses PrimeKG drug–gene target edges from the previous export and leaves the colorectal graph files intact.

Open this notebook beside your `kg_data` folder. If needed, run `%pip install pandas openpyxl torch` in a separate cell. Set `graph_dir` to the folder containing your existing `nodes.csv` and `edges.csv`.

Start with the observed lapatinib comparison and the assay coverage table, then check whether a subtype-aware model improves prediction on **entire held-out cell lines**. The same screened drugs can appear in train and test. This is a preclinical response benchmark for known compounds, not a clinical prediction or a test on novel compounds.

In [2]:
%pip install "openpyxl>=3.1.5"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import re
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import torch
from torch import nn

project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "agent/config.py").is_file() and (p / "src/primekg").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("Start Jupyter within the BioHack2026 checkout.")

graph_dir = project_root / "data/processed/primekg_ecograph"
data_dir = project_root / "artifacts/predictions/breast"
data_dir.mkdir(parents=True, exist_ok=True)
gdsc_path = project_root / "data/external/gdsc/GDSC1_fitted_dose_response_27Oct23.xlsx"
gdsc_path.parent.mkdir(parents=True, exist_ok=True)
gdsc_url = "https://cog.sanger.ac.uk/cancerrxgene/GDSC_release8.5/" + gdsc_path.name
if not gdsc_path.is_file():
    temporary = gdsc_path.with_suffix(".part")
    try:
        urlretrieve(gdsc_url, temporary)
        temporary.replace(gdsc_path)
    except OSError as error:
        raise RuntimeError(f"Download GDSC1 manually from Cell Model Passports and save it to {gdsc_path}") from error
    finally:
        temporary.unlink(missing_ok=True)

# Curated panel based on published cell-line receptor classifications.
# HER2-positive includes models with different hormone receptor backgrounds.
groups = {
    "HER2-positive": ["AU565", "BT474", "EFM192A", "HCC1419", "HCC1569", "HCC1954",
                      "HCC202", "HCC2218", "MDAMB361", "SKBR3", "UACC893", "ZR7530"],
    "Triple-negative": ["BT20", "BT549", "HCC38", "HCC70", "HCC1143", "HCC1187",
                        "HCC1806", "HCC1937", "HS578T", "MDAMB157", "MDAMB231",
                        "MDAMB436", "MDAMB468", "SUM149PT", "SUM159PT"],
}
def line_key(value):
    return re.sub(r"[^A-Z0-9]", "", str(value).upper())

subtype_by_line = {line_key(line): group for group, lines in groups.items() for line in lines}
fields = ["CELL_LINE_NAME", "DRUG_ID", "DRUG_NAME", "AUC", "LN_IC50"]
screen = pd.read_excel(gdsc_path, usecols=fields, engine="openpyxl")
screen["line"] = screen["CELL_LINE_NAME"].map(line_key)
screen["subtype"] = screen["line"].map(subtype_by_line)
screen = screen.dropna(subset=["subtype", "DRUG_ID", "DRUG_NAME", "AUC"]).copy()
assert screen["AUC"].between(0, 1).all(), "Unexpected GDSC AUC range"
screen["drug_key"] = screen["DRUG_NAME"].str.strip().str.casefold()
# If a name maps to multiple screened IDs, leave it out rather than combine compounds.
unique_names = screen.groupby("drug_key")["DRUG_ID"].nunique()
screen = screen[screen["drug_key"].isin(unique_names[unique_names.eq(1)].index)]
response = screen.groupby(["line", "subtype", "DRUG_ID", "DRUG_NAME", "drug_key"], as_index=False).agg(
    auc=("AUC", "median"), ln_ic50=("LN_IC50", "median")
)
display(response.groupby("subtype")["line"].nunique().rename("screened_cell_lines"))

subtype
HER2-positive      11
Triple-negative    13
Name: screened_cell_lines, dtype: int64

In [4]:
# Observed benchmark. Lower AUC means greater sensitivity in the assay.
benchmark_drug = "lapatinib"
observed = response[response["drug_key"].eq(benchmark_drug)]
if observed["subtype"].nunique() == 2:
    display(observed[["subtype", "line", "auc", "ln_ic50"]].sort_values(["subtype", "auc"]))
    display(observed.groupby("subtype")["auc"].agg(["size", "median", "min", "max"]))
    if observed.groupby("subtype")["line"].nunique().min() < 4:
        print("Sparse lapatinib coverage. Treat this as a descriptive check.")
else:
    print("Lapatinib was not measured in both groups in this file.")

# Related inhibitors are candidates for a better-covered assay comparison.
# Report all prespecified candidates and their coverage before inspecting outcomes.
her_inhibitors = ["lapatinib", "afatinib", "neratinib", "sapitinib"]
coverage = (
    response[response["drug_key"].isin(her_inhibitors)]
    .groupby(["drug_key", "subtype"])["line"].nunique()
    .unstack(fill_value=0)
    .reindex(index=her_inhibitors, columns=list(groups), fill_value=0)
)
display(coverage.rename_axis("HER-family inhibitor"))

,subtype,line,auc,ln_ic50
3879,HER2-positive,HCC2218,0.418857,-2.692161
631,HER2-positive,BT474,0.654335,-1.011539
1728,Triple-negative,HCC1187,0.988593,2.957433


,size,median,min,max
subtype,,,,
HER2-positive,2,0.536596,0.418857,0.654335
Triple-negative,1,0.988593,0.988593,0.988593


Sparse lapatinib coverage. Treat this as a descriptive check.


subtype,HER2-positive,Triple-negative
HER-family inhibitor,,
lapatinib,2,1
afatinib,0,0
neratinib,0,0
sapitinib,0,0


In [5]:
# Map measured compounds to unique PrimeKG drug IDs using exact normalized names.
nodes = pd.read_csv(graph_dir / "nodes.csv", dtype={"node_id": "string"}, low_memory=False)
assert nodes["node_index"].is_unique
kind = nodes.set_index("node_index")["node_type"]
kg_drugs = nodes.loc[nodes["node_type"].eq("drug"), ["node_index", "node_id", "node_name"]].copy()
kg_drugs["drug_key"] = kg_drugs["node_name"].str.strip().str.casefold()
kg_drugs = kg_drugs.loc[~kg_drugs["drug_key"].duplicated(keep=False)]
parts = []
for chunk in pd.read_csv(graph_dir / "edges.csv", usecols=["x_index", "y_index", "relation"],
                         chunksize=250_000):
    parts.append(chunk.loc[chunk["relation"].eq("drug_protein"), ["x_index", "y_index"]])
target_rows = pd.concat(parts, ignore_index=True)
forward = target_rows["x_index"].map(kind).eq("drug")
targets = pd.DataFrame({
    "drug": target_rows["x_index"].where(forward, target_rows["y_index"]),
    "gene": target_rows["y_index"].where(forward, target_rows["x_index"]),
}).drop_duplicates()
assert targets["drug"].map(kind).eq("drug").all()
assert targets["gene"].map(kind).eq("gene/protein").all()

map_audit = response[["DRUG_ID", "DRUG_NAME", "drug_key"]].drop_duplicates().merge(
    kg_drugs, on="drug_key", how="left", validate="m:1"
)
map_audit["has_targets"] = map_audit["node_index"].isin(targets["drug"])
map_audit.to_csv(data_dir / "drug_mapping_audit.csv", index=False)
mapping = map_audit.loc[map_audit["has_targets"], ["drug_key", "node_index", "node_id"]]
assert mapping["drug_key"].is_unique
model_data = response.merge(mapping, on="drug_key", validate="m:1")
print(f"Mapped {model_data['drug_key'].nunique()} of {response['drug_key'].nunique()} screened drugs with target edges.")
assert not model_data.empty, "No GDSC compounds matched to PrimeKG drug target edges."

Mapped 72 of 354 screened drugs with target edges.


In [6]:
# Hold out whole cell lines from both groups. Include lapatinib-measured lines in
# each split where possible, using only assay availability to form the split.
seed = 42
rng = np.random.default_rng(seed)
test_lines = set()
for subtype, group in model_data.groupby("subtype"):
    lines = sorted(group["line"].unique())
    if len(lines) < 4:
        raise ValueError(f"Need at least four mapped cell lines in {subtype}")
    measured = sorted(group.loc[group["drug_key"].eq(benchmark_drug), "line"].unique())
    test_size = max(2, round(0.30 * len(lines)))
    if len(measured) >= 2:
        held, kept = rng.choice(measured, size=2, replace=False)
        pool = sorted(set(lines) - {held, kept})
        test_lines.update([held, *rng.choice(pool, size=test_size - 1, replace=False)])
    else:
        test_lines.update(rng.choice(lines, size=test_size, replace=False))
train = model_data.loc[~model_data["line"].isin(test_lines)].copy()
test = model_data.loc[model_data["line"].isin(test_lines)].copy()
assert set(train["line"]).isdisjoint(test["line"])
# Compare group rankings on drugs measured in at least two training lines per group.
train_counts = train.groupby(["node_index", "subtype"])["line"].nunique().unstack(fill_value=0)
drugs = pd.Index(train_counts.index[train_counts.ge(2).all(axis=1)])
train = train[train["node_index"].isin(drugs)].copy()
test = test[test["node_index"].isin(drugs)].copy()
assert len(drugs) >= 5 and test["subtype"].nunique() == 2, "Insufficient shared screen coverage."
display(pd.DataFrame({"train_lines": train.groupby("subtype")["line"].nunique(),
                      "test_lines": test.groupby("subtype")["line"].nunique()}))
print("Shared candidate drugs:", len(drugs))

,train_lines,test_lines
subtype,,
HER2-positive,8,3
Triple-negative,9,4


Shared candidate drugs: 60


In [7]:
# One graph message-passing step pools each drug's PrimeKG target embeddings.
# Compare a subtype-aware decoder with the same architecture given constant subtype input.
torch.set_num_threads(2)
target_sets = targets[targets["drug"].isin(drugs)].groupby("drug")["gene"].agg(list)
gene_ids = pd.Index(sorted({gene for neighbors in target_sets for gene in neighbors}))
neighbors = [gene_ids.get_indexer(target_sets.loc[drug]) for drug in drugs]
flat = torch.as_tensor(np.concatenate(neighbors), dtype=torch.long)
offsets = torch.as_tensor(np.r_[0, np.cumsum([len(group) for group in neighbors])], dtype=torch.long)
drug_means = train.groupby("node_index")["auc"].mean().reindex(drugs)
prior = torch.tensor(drug_means.to_numpy(), dtype=torch.float32)

def tensors(frame):
    positions = drugs.get_indexer(frame["node_index"])
    assert (positions >= 0).all()
    return (
        torch.tensor(positions, dtype=torch.long),
        torch.tensor(frame["subtype"].eq("HER2-positive").to_numpy(dtype=float), dtype=torch.float32),
    )

class ResponseGNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.targets = nn.EmbeddingBag(len(gene_ids), 32, mode="mean", include_last_offset=True)
        self.decoder = nn.Sequential(nn.Linear(33, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, drug_positions, her2):
        target_vectors = self.targets(flat, offsets)[drug_positions]
        features = torch.cat([target_vectors, her2[:, None]], dim=1)
        return prior[drug_positions] + self.decoder(features).squeeze(1)

train_drug, train_her2 = tensors(train)
train_y = torch.tensor(train["auc"].to_numpy(), dtype=torch.float32)

def fit(use_subtype):
    torch.manual_seed(seed)
    model = ResponseGNN()
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=0.01)
    her2 = train_her2 if use_subtype else torch.full_like(train_her2, 0.5)
    for _ in range(75):
        optimizer.zero_grad()
        loss = nn.functional.mse_loss(model(train_drug, her2), train_y)
        loss.backward()
        optimizer.step()
    return model.eval()

subtype_model = fit(use_subtype=True)
shared_model = fit(use_subtype=False)

@torch.no_grad()
def predict(model, frame, use_subtype):
    drug_pos, her2 = tensors(frame)
    if not use_subtype:
        her2 = torch.full_like(her2, 0.5)
    return model(drug_pos, her2).clamp(0, 1).numpy()

test["pooled_drug_mean"] = test["node_index"].map(drug_means)
group_mean = train.groupby(["node_index", "subtype"])["auc"].mean().rename("subtype_drug_mean")
test = test.join(group_mean, on=["node_index", "subtype"], validate="m:1")
test["shared_input_gnn"] = predict(shared_model, test, use_subtype=False)
test["subtype_gnn"] = predict(subtype_model, test, use_subtype=True)
models = ["pooled_drug_mean", "subtype_drug_mean", "shared_input_gnn", "subtype_gnn"]
errors = test[models].sub(test["auc"], axis=0).abs()
# Give each held-out cell line equal weight.
display(errors.groupby(test["line"]).mean().mean().sort_values().rename("test_MAE"))
test.to_csv(data_dir / "heldout_predictions.csv", index=False)

subtype_drug_mean    0.064325
subtype_gnn          0.065197
shared_input_gnn     0.065828
pooled_drug_mean     0.065831
Name: test_MAE, dtype: float64

In [8]:
# Compare measured and predicted lapatinib response on held-out lines.
anchor = test.loc[test["drug_key"].eq(benchmark_drug)]
if anchor["subtype"].nunique() == 2:
    display(anchor.groupby("subtype").agg(
        cell_lines=("line", "nunique"), measured_auc=("auc", "mean"),
        predicted_auc=("subtype_gnn", "mean"), shared_input_auc=("shared_input_gnn", "mean")
    ))
else:
    print("Lapatinib lacks a two-group held-out model comparison after mapping and coverage filters.")
    print("The measured comparison above remains available.")

catalog = kg_drugs.set_index("node_index").loc[drugs, ["node_name", "node_id"]].reset_index()
all_rankings = []
for subtype in groups:
    rows = catalog.assign(subtype=subtype).copy()
    rows["predicted_auc"] = predict(subtype_model, rows, use_subtype=True)
    rows["shared_input_auc"] = predict(shared_model, rows, use_subtype=False)
    rows = rows.sort_values(["predicted_auc", "node_index"], ignore_index=True)
    rows["rank"] = rows.index + 1
    all_rankings.append(rows)
rankings = pd.concat(all_rankings, ignore_index=True)
rankings.to_csv(data_dir / "breast_subtype_rankings.csv", index=False)
top20 = rankings[rankings["rank"].le(20)]
display(top20[["subtype", "rank", "node_name", "node_id", "predicted_auc"]])
top_sets = [set(group["node_index"]) for _, group in top20.groupby("subtype")]
print("Shared drugs in the two top lists:", len(top_sets[0] & top_sets[1]))

Lapatinib lacks a two-group held-out model comparison after mapping and coverage filters.
The measured comparison above remains available.


,subtype,rank,node_name,node_id,predicted_auc
0,HER2-positive,1,Panobinostat,DB06603,0.439094
1,HER2-positive,2,Belinostat,DB05015,0.553466
2,HER2-positive,3,Bleomycin,DB00290,0.566028
3,HER2-positive,4,AT-7519,DB08142,0.593494
4,HER2-positive,5,Tipifarnib,DB04960,0.649373
5,HER2-positive,6,Tanespimycin,DB05134,0.666903
6,HER2-positive,7,Vinblastine,DB00570,0.676630
7,HER2-positive,8,Trichostatin A,DB04297,0.679305
8,HER2-positive,9,Pelitinib,DB05524,0.683118
9,HER2-positive,10,Vinorelbine,DB00361,0.703607


Shared drugs in the two top lists: 17


### Interpreting the result

Compare the subtype-aware model with the shared-input GNN and the simple subtype drug mean. A changed ranking is useful only if it agrees with held-out response. AUC in these files measures **cell-line drug sensitivity**, and lower values indicate stronger activity in the assay. Rankings apply to drugs measured in both training groups. A single split on a small curated panel is exploratory.

Sources

- [Sanger GDSC fitted drug response](https://cellmodelpassports.sanger.ac.uk/downloads/gdsc)
- [HER2-positive model annotations](https://pmc.ncbi.nlm.nih.gov/articles/PMC5927625/) and [additional HER2-positive models](https://pmc.ncbi.nlm.nih.gov/articles/PMC7039866/)
- [Triple-negative models](https://pmc.ncbi.nlm.nih.gov/articles/PMC4297247/) and [additional TNBC models](https://pmc.ncbi.nlm.nih.gov/articles/PMC6746778/)
- [PrimeKG](https://github.com/mims-harvard/PrimeKG)